# LoRA vs KronA — Unsloth

Fine-tune a small chat model **twice** — once with **LoRA**, once with **KronA** — then
compare them apples-to-apples.

> **KRONA = KronA** = the Kronecker adapter (arXiv:2212.10650). In HuggingFace PEFT it is the
> **`LoKr`** adapter (`LoKrConfig`) — that is what we train.

All the logic lives in the **`krona/` package**, so this notebook is only a thin driver and
`python run.py` does exactly the same thing. Gotchas and decisions: `docs/NOTES.md`.

> **One hard constraint:** KronA needs **16-bit** (`load_in_4bit=False`). PEFT's LoKr does
> `weight.reshape(base_layer.weight.shape)`, and on a bitsandbytes 4-bit layer that shape is
> the *nibble-packed* storage, so it fails with
> `shape '[589824, 1]' is invalid for input of size 1179648`. **LoKr is not QLoRA-compatible.**
> Both arms run 16-bit so the comparison stays fair. On a bigger GPU, raise the **model**
> (e.g. `gemma-3-4b-it`), not the precision.

## 1. Install

Run once per session.

In [ ]:
# 1) Install (run once per session)
!pip install -q -r requirements.txt

## 2. Run the experiment

Environment check → dataset → LoRA + KronA → comparison table, samples and `results.json`.

Outputs go to `outputs/`, or `/workspace/outputs` on RunPod (a network volume survives a pod
restart). Anything you set below can also live in `.env` as `KRONA_*` variables.

In [ ]:
# 2) Everything else lives in krona/ — this is the only call
from krona.pipeline import run
from krona.config import Config

cfg = Config.from_env()
cfg.max_samples = 1000     # comment these out to use .env / defaults
cfg.steps       = 60
# cfg.model     = "unsloth/gemma-3-4b-it"   # bigger GPU? raise the model, not the precision

results = run(cfg)